In [ ]:
import sys
import os
import json
import numpy as np
import scipy as sp
import pandas as pd


sys.path.append(os.path.join(os.getcwd(), '../../src/'))
from utils_mouse import read_utils

sys.path.append(os.path.join(os.getcwd(), '../../../../'))
from utils import plot_utils
from utils import train_utils
from models import model_utils

In [ ]:
import importlib
importlib.reload(read_utils)
importlib.reload(plot_utils)
importlib.reload(train_utils)

In [ ]:
ll = 'll00'
data_variant = 'gcamp'
model_variant = 'ss'
organism = 'mouse-2019-stringer'

In [ ]:
file_id = 2

neuron_labels = ['02_425', '02_7412', '02_7331', '02_6586', '02_1408', '02_6449', '02_7204', '02_8167', '02_3761', '02_2988']

In [ ]:
manual = False

if manual:
	with open('../../hyperparams_manual.json', 'r') as f:
		reg_lambdas = json.load(f)
else:
	with open('../../hyperparams.json', 'r') as f:
		reg_lambdas = json.load(f)['geometric'][ll]

bom_folder = (f'{data_variant}_bom_{model_variant}_'
			  + '_'.join([f'{param.split('_')[0]}{reg_lambdas[model_variant]['bom'][param]:.2E}' for param in reg_lambdas[model_variant]['bom'].keys()]))
hgm_folder = (f'{data_variant}_hgm_{model_variant}_'
			  + '_'.join([f'{param.split('_')[0]}{reg_lambdas[model_variant]['hgm'][param]:.2E}' for param in reg_lambdas[model_variant]['hgm'].keys()]))

In [ ]:
path = os.path.join(read_utils.get_data_path(), '6163622/sessions')

(filename,
 traces,
 continuous_behaviors,
 motion_pcs,
 motion_masks) = read_utils.get_data(path, file_id, with_whisking=True)

# clear nan values
# mask = motion_pcs[0].notna().all(axis=1)
mask = continuous_behaviors.notna().all(axis=1)

traces = traces[mask]
motion_pcs[0] = motion_pcs[0][mask]
motion_pcs[1] = motion_pcs[1][mask]
continuous_behaviors = continuous_behaviors[mask]

# naive ID unknown neurons
traces.columns = pd.Index([
	f'{file_id:02d}_{n_id:03d}' if empty or type(label) is not str else label for n_id, (label, empty) in
	enumerate(zip(traces.columns, traces.columns == ''))
])

# preprocess
def reconstruct_calcium(spikes, fs, decay=2.0):
	dt = 1.0 / fs
	gamma = np.exp(-dt / decay)
	trace = np.zeros(spikes.shape, dtype=np.float32)
	for t in range(1, spikes.shape[0]):
		trace[t, :] = gamma * trace[t - 1, :] + spikes[t, :]
	return trace

sampling_rate = read_utils.get_sampling_rate(os.path.join(path, filename))
traces = pd.DataFrame(reconstruct_calcium(traces.values, fs=sampling_rate), columns=traces.columns)
classes = pd.DataFrame(
	continuous_behaviors[['running_speed']].values > np.quantile(continuous_behaviors[['running_speed']].values, 0.05),
	columns=['run']
)

In [ ]:
neuron_labels = [label for label in neuron_labels if label in traces.columns]

# manifold

In [ ]:
_, bom_man, bom_rec, bom_res, bom_pres, bom_resss, bom_bss = train_utils.load_predictions(os.path.join(read_utils.get_model_path(), 'predictions', bom_folder), filename)
_, hgm_man, hgm_rec, hgm_res, hgm_pres, hgm_resss, hgm_bss = train_utils.load_predictions(os.path.join(read_utils.get_model_path(), 'predictions', hgm_folder), filename)

bom_params = train_utils.load_params(os.path.join(f'{read_utils.get_model_path()}', 'params', bom_folder), filename)
hgm_params = train_utils.load_params(os.path.join(f'{read_utils.get_model_path()}', 'params', hgm_folder), filename)

In [ ]:
bom_manifolds = list()
bom_manifold_vars_expl = list()
for s in range(len(bom_params)):
	decode = lambda X: X @ model_utils.orthonormalize_qr(bom_params[s]['model']['UN']).T
	manifold, var_explained = train_utils.postprocess_manifold(bom_man[s].values, traces.values, decode, scale=False, align=True)
	bom_manifolds.append(pd.DataFrame(data=manifold, columns=bom_man[s].columns))
	bom_manifold_vars_expl.append(var_explained)

hgm_manifolds = list()
hgm_manifold_vars_expl = list()
for s in range(len(hgm_params)):
	decode = lambda X: X @ model_utils.orthonormalize_qr(hgm_params[s]['model']['UN']).T
	manifold, var_explained = train_utils.postprocess_manifold(hgm_man[s].values, traces.values, decode, scale=False, align=True)
	hgm_manifolds.append(pd.DataFrame(data=manifold, columns=hgm_man[s].columns))
	hgm_manifold_vars_expl.append(var_explained)

In [ ]:
plot_utils.plot_manifolds_gradient([f'{str(file_id)}: {filename} s{s}' for s in range(len(bom_manifolds))],
						  [bom_manifolds[s] for s in range(len(bom_manifolds))],
						  [continuous_behaviors[['running_speed']] for _ in range(len(bom_manifolds))],
						  bom_manifold_vars_expl, None,
						  title='bom manifold', plots_per_row=len(bom_manifolds), top_space=3.5,
						  save_path=None)

In [ ]:
plot_utils.plot_manifolds_gradient([f'{str(file_id)}: {filename} s{s}' for s in range(len(hgm_manifolds))],
						  [hgm_manifolds[s] for s in range(len(hgm_manifolds))],
						  [continuous_behaviors[['running_speed']] for _ in range(len(bom_manifolds))],
						  hgm_manifold_vars_expl, None,
						  title='hgm manifold', plots_per_row=len(hgm_manifolds), top_space=3.5,
						  save_path=None)

In [ ]:
sigmoid = lambda x: 1 / (1 + np.exp(-x))

resss_ids = [hgm_resss[0].columns.get_loc(ss) for ss in hgm_resss[0].columns]

splits = 5

for resss_id in resss_ids:
	sigmoid_terms = list()
	manifold_terms = list()
	vars_explained = list()
	for split in range(splits):
		hgm_man_zs = (hgm_man[split].values - hgm_man[split].values.mean(axis=0)) / np.where((N_std := hgm_man[split].values.std(axis=0)) == 0.0, 1.0, N_std)
		WM = hgm_params[split]['model']['WM'][:, resss_id]
		sigmoid_term = sigmoid(hgm_man_zs @ WM)
		sigmoid_terms.append(pd.DataFrame(data=sigmoid_term))

		decode = lambda X: X @ model_utils.orthonormalize_qr(hgm_params[split]['model']['UN']).T
		manifold_term, var_explained = train_utils.postprocess_manifold(hgm_man[split].values, traces.values, decode, scale=False, align=True)
		manifold_terms.append(pd.DataFrame(data=manifold_term, columns=hgm_man[split].columns))
		vars_explained.append(var_explained)

	plot_utils.plot_manifolds_gradient([f'{str(file_id)}: {filename} s{s}' for s in range(splits)],
							  [manifold_terms[s] for s in range(splits)],
							  [sigmoid_terms[s] for s in range(splits)],
							  vars_explained, None,
							  title=f'gcamp manifolds by gating of {hgm_resss[0].columns[resss_id]}', plots_per_row=splits, top_space=1,
							  save_path=None)

# traces

In [ ]:
# max_x = 500
max_x = traces.values.shape[0]

plot_utils.plot_traces_splits(
	[hgm_rec[i].loc[:max_x, neuron_labels] for i in range(len(hgm_rec))],
	[bom_rec[i].loc[:max_x, neuron_labels] for i in range(len(hgm_rec))],
	traces.loc[:max_x, neuron_labels],
	classes.loc[:max_x, ['run']], label='running',
	title=f'{filename} gcamp hgm & bom definitions manifold', ylimits=None, plots_per_row=1,
	save_path=None)  #'../../out/plots/GCAMP_MAN'

In [ ]:
# max_x = 500
max_x = traces.values.shape[0]

plot_utils.plot_traces_splits(
	[hgm_res[i].loc[:max_x, neuron_labels] for i in range(splits)],
	[bom_res[i].loc[:max_x, neuron_labels] for i in range(splits)],
	traces.loc[:max_x, neuron_labels],
	classes.loc[:max_x, ['run']], label='running',
	title=f'{filename} gcamp hgm & bom definitions residuals', ylimits=None, plots_per_row=1,
	save_path=None)  #'../../out/plots/GCAMP_RES'

In [ ]:
# max_x = 500
max_x = traces.values.shape[0]

for i in range(splits):
	plot_utils.plot_traces_splits_pairs(
		[hgm_res[i].loc[:max_x, neuron_labels],
		 hgm_pres[i].loc[:max_x, neuron_labels]],
		[bom_res[i].loc[:max_x, neuron_labels],
		 bom_pres[i].loc[:max_x, neuron_labels]],
		traces.loc[:max_x, neuron_labels],
		classes.loc[:max_x, ['run']], label='running',
		title=f'{filename} split {i} gcamp hgm & bom predictions residuals', ylimits=None, plots_per_row=1,
		save_path=None)  #../../out/plots/f'GCAMP_PRES_S{i}'

In [ ]:
# max_x = 500
max_x = traces.values.shape[0]

plot_utils.plot_traces_splits(
	[hgm_bss[i].loc[:max_x] for i in range(splits)][0:1],
	[bom_bss[i].loc[:max_x] for i in range(splits)][0:1],
	None,
	classes.loc[:max_x, ['run']], label='running',
	title=f'{filename} gcamp hgm & bom definitions residuals', ylimits=None, plots_per_row=1,
	save_path=None)  #'../../out/plots/GCAMP_RES'

In [ ]:
# max_x = 500
max_x = traces.values.shape[0]

plot_utils.plot_traces_splits(
	[hgm_resss[i].loc[:max_x] for i in range(splits)][0:1],
	[bom_resss[i].loc[:max_x] for i in range(splits)][0:1],
	None,
	classes.loc[:max_x, ['run']], label='running',
	title=f'{filename} gcamp hgm & bom definitions residuals', ylimits=None, plots_per_row=1,
	save_path=None)  #'../../out/plots/GCAMP_RES'